In [2]:
import os

import numpy as np
import pandas as pd
import pytorch_lightning as pl
import scanpy as sc
import torch

import T_perturb
from T_perturb.Dataloaders.datamodule import CellGenDataModule
from T_perturb.Perturb.trainer import PerturberGenerationTrainer, PerturberInferenceTrainer
from T_perturb.src.utils import label_encoder
from T_perturb.tests.test_cellgen_training import dummy_dataset
from T_perturb.tests.test_countdecoder_training import dummy_cell_gene_matrix

In [3]:
import sys
print("\nKernel directory path:")
print(os.path.dirname(sys.executable))


Kernel directory path:
/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/bin


In [4]:
if os.getcwd().split('/')[-1] != 'healthy_imm_expr':
    # set working directory to root of repository
    os.chdir('/lustre/scratch126/cellgen/team361/kl11/t_generative/')

In [5]:
tgt_vocab_size = 101  # +1 for padding token
num_samples = 100
num_genes = 100
max_seq_length = 50
n_total_tps = 2
num_samples = 100
batch_size = 4
pred_tps = [1, 2]
context_tps = [1, 2]
d_model = 12

genes_to_perturb = [70, 41]
perturbation_token = 0


In [6]:
src_counts = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
)
src_dataset = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
)
tgt_counts_dict = dummy_cell_gene_matrix(
    num_cells=num_samples,
    num_genes=num_genes,
    total_time_steps=n_total_tps,
)
tgt_datasets = dummy_dataset(
    max_len=max_seq_length,
    vocab_size=tgt_vocab_size,
    num_samples=100,
    total_time_steps=n_total_tps,
)

In [7]:
conditions = None
condition_keys = None
conditions_combined = None

In [8]:
if condition_keys is None:
    condition_keys = 'tmp_batch'
    # create a mock vector if there are no batch effect
    tmp_series = pd.DataFrame(
        {
            condition_keys: np.ones(num_samples),
        }
    )


In [9]:
if isinstance(condition_keys, str):
    condition_keys_ = [condition_keys]
else:
    condition_keys_ = condition_keys

if conditions is None:
    if condition_keys is not None:
        conditions_ = {}
        for cond in condition_keys_:
            conditions_[cond] = tmp_series[cond].unique().tolist()
    else:
        conditions_ = {}
else:
    conditions_ = conditions

if conditions_combined is None:
    if len(condition_keys_) > 1:
        tmp_series['conditions_combined'] = tmp_series[condition_keys].apply(
            lambda x: '_'.join(x), axis=1
        )
    else:
        tmp_series['conditions_combined'] = tmp_series[condition_keys]
    conditions_combined_ = tmp_series['conditions_combined'].unique().tolist()
else:
    conditions_combined_ = conditions_combined

condition_encodings = {
    cond: {k: v for k, v in zip(conditions_[cond], range(len(conditions_[cond])))}
    for cond in conditions_.keys()
}
conditions_combined_encodings = {
    k: v for k, v in zip(conditions_combined_, range(len(conditions_combined_)))
}

In [10]:
tgt_adata_tmp = sc.AnnData(X=tgt_counts_dict['tgt_h5ad_t1'].squeeze(), obs=tmp_series)

/lustre/scratch126/cellgen/team361/kl11/t_generative/.cellgen_4096/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [11]:
if (condition_encodings is not None) and (condition_keys_ is not None):
    conditions = [
        label_encoder(
            tgt_adata_tmp,
            encoder=condition_encodings[condition_keys_[i]],
            condition_key=condition_keys_[i],
        )
        for i in range(len(condition_encodings))
    ]
    conditions = torch.tensor(conditions, dtype=torch.long).T
    conditions_combined = label_encoder(
        tgt_adata_tmp,
        encoder=conditions_combined_encodings,
        condition_key='conditions_combined',
    )
    conditions_combined = torch.tensor(conditions_combined, dtype=torch.long)


In [21]:
trainer_params = {
    'tgt_vocab_size': tgt_vocab_size,
    'd_model': d_model,
    'num_heads': 4,
    'num_layers': 1,
    'd_ff': 8,
    'max_seq_length': max_seq_length + 10,
    'dropout': 0.0,
    'pred_tps': pred_tps,
    'context_tps': context_tps,
    'n_total_tps': n_total_tps,
    'precision': 'high',
    'mask_scheduler': 'pow',
    'output_dir': './T_perturb/T_perturb/tests/res',
    'encoder': 'Transformer_encoder',
    'var_list': None,
    'genes_to_perturb': genes_to_perturb,
    'perturbation_token': perturbation_token,
    'context_mode': True,
    'perturbation_mode': ['src', 'tgt'],
}
decoder_module = PerturberInferenceTrainer(
    weight_decay=0.0,
    end_lr=1e-3,
    return_embeddings=True,
    **trainer_params
    )
# decoder_module = PerturberGenerationTrainer(
#     ckpt_masking_path='./T_perturb/T_perturb/tests/checkpoints/baseline_masking_checkpoint-epoch=00.ckpt',
#     ckpt_count_path='./T_perturb/T_perturb/tests/checkpoints/baseline_counts_checkpoint-epoch=00.ckpt',
#     loss_mode='zinb',
#     lr=1e-3,
#     weight_decay=0.0,
#     conditions=conditions_,
#     conditions_combined=conditions_combined_,
#     temperature=1.5,
#     iterations=19,
#     sequence_length=max_seq_length - 10,
#     n_samples=3,
#     seed=42,
#     n_genes=num_genes,
#     **trainer_params
# )

data_module = CellGenDataModule(
    src_counts=src_counts,
    tgt_counts_dict=tgt_counts_dict,
    src_dataset=src_dataset,
    tgt_datasets=tgt_datasets,
    batch_size=batch_size,
    num_workers=1,
    pred_tps=pred_tps,
    context_tps=context_tps,
    n_total_tps=n_total_tps,
    train_indices=None,
    test_indices=np.random.choice(100, 20, replace=False),
    max_len=max_seq_length,
    condition_keys=condition_keys_,
    condition_encodings=condition_encodings,
    conditions=conditions,
    conditions_combined=conditions_combined,
)
data_module.setup()
test_loader = data_module.test_dataloader()

Using CPU for training
start training
cls_token_1 tensor([101])
cls_token_2 tensor([102])
test kwargs {'weight_decay': 0.0, 'end_lr': 0.001, 'return_embeddings': True, 'tgt_vocab_size': 101, 'd_model': 12, 'num_heads': 4, 'num_layers': 1, 'd_ff': 8, 'max_seq_length': 60, 'dropout': 0.0, 'pred_tps': [1, 2], 'context_tps': [1, 2], 'n_total_tps': 2, 'precision': 'high', 'mask_scheduler': 'pow', 'output_dir': './T_perturb/T_perturb/tests/res', 'encoder': 'Transformer_encoder', 'var_list': None, 'context_mode': True}
Start datamodule


In [20]:
import importlib
import T_perturb.src.utils
importlib.reload(T_perturb.Perturb.trainer)
importlib.reload(T_perturb.src.utils)
from T_perturb.Perturb.trainer import PerturberInferenceTrainer

In [22]:
trainer = pl.Trainer(
    limit_test_batches=1,  # Limit to a single batch for quick testing
    logger=False,
)
trainer.test(decoder_module, data_module)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_test_batches=1)` was configured so 1 batch will be used.


Testing DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]perturbating tgt
perturbating tgt
perturbating src
true dict_keys(['dec_embedding', 'self_attn_weights', 'cross_attn_weights', 'dec_logits', 'labels', 'selected_time_step', 'mean_embedding'])
true {'dec_embedding': {1: tensor([[[-1.1728, -0.3078, -0.5666,  ...,  1.8597, -1.5542,  0.5722],
         [ 0.1702, -0.2434, -1.3983,  ...,  0.5976, -0.5852,  0.2605],
         [ 1.3001, -1.0311, -0.7101,  ...,  0.6765, -1.7890, -0.2728],
         ...,
         [-0.4639, -1.2336, -0.9570,  ...,  2.1127, -0.9096,  0.6283],
         [-0.6234, -0.5731, -1.1603,  ...,  2.1075, -1.0519,  0.7138],
         [-0.1986, -0.1375, -1.3775,  ...,  2.0293, -1.2251,  0.7062]],

        [[-1.1052, -0.2659, -0.5476,  ...,  1.9458, -1.5792,  0.5526],
         [ 0.6277, -1.8837, -0.3254,  ...,  1.3945, -1.3695,  0.8493],
         [ 0.5116, -0.9725, -0.4589,  ...,  2.1707, -1.2803, -0.1774],
         ...,
         [-0.4121, -1.2216, -0.9051,  ...,  2.1841, -

[{}]